#### Goal:
- To create a dataobject to have all information about a CDD Case risk categories for all W&R Sourcesystems

#### Author
- Devi.Chennareddy@rabobank.com
- Abhishek.R.Jaiswal@rabobank.com

#### Flow Of Logic

- Take GCOB, KN1, Legacy2 & NLSVF cases, along with their risk categories from GDP objects.
- Unioning all the case risk categories information from different sourcesystems
- Save the final output in saradar storage account

##### Version & Changes

|     Developer |Date	   | PBI/Bug No |	Changes done |
|----------|----------|----------|----------|
| Abhishek Jaiswal	 |25-July-2025 |13218266 |KN1 Risk calculation based on Barend's solution		
| Abhishek Jaiswal	 |01-Aug-2025 |13277933 |Adding Material risk
| Abhishek Jaiswal	 |14-Aug-2025 |13420151 |Bug fix Material risk
| Mahalakshmi V | 20-Nov-2025 | 14165735 | Made notebook compatible for Historical Load
| Prajit Tatari	 |11-Feb-2026 |15058674 |Adding MDM RANZ data from GDP  | 2
|Rhea Gupta | 30-Mar-2026 |15580816 | Included RANZ in version 3
| Mahalakshmi V | 20-May-2026 | 16071492 | Included RANZ V4


In [0]:
import os
import pandas as pd
from datetime import datetime, timedelta
from pyspark.dbutils import DBUtils
from pyspark.sql import SparkSession

In [0]:
import os
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
# RANZ_ReadStorage = os.environ['AU_GDP_Defined_Storage_Account']
environment=os.environ['ENV']
radar_datamodel_version_number=3
NLS_ReadStorage = os.environ['GDP_NA_STORAGE_NAME']

In [0]:
from RadarUtils import *

In [0]:
SARADAR='saradar'+environment

In [0]:
authenticate_storage_account(ReadStorage)
authenticate_storage_account(GIC_ReadStorage)
authenticate_storage_account('edlcorestdauprod0001') #RANZ_ReadStorage
authenticate_storage_account(NLS_ReadStorage)
authenticate_storage_account(SARADAR)

In [0]:
dbutils.widgets.text("Load_Date", "")
dbutils.widgets.text("RunType", "daily")  # default is daily

Load_Date = dbutils.widgets.get("Load_Date")
RunType = dbutils.widgets.get("RunType").lower()
if Load_Date:
    RunType="historical"
    
if not Load_Date:
    Load_Date = datetime.today().strftime('%Y%m%d')

print(f"Running for Load Date: {Load_Date}")

In [0]:
party_CDDCases_RiskCategory_dataobject = 'Party_CDDCase_RiskCategories'

In [0]:
load_dts = f"EDL_LOAD_DTS={Load_Date}*"
print (load_dts)

In [0]:
# List of dataobjects from GDP
load_df =[
'Party_RiskModelInstanceQuestionAnswers'
, 'Party_RiskModelCategories'
, 'Party_RiskModelInstance'
, 'party_WRCDDModel'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='RISKMODEL',Load_Date=Load_Date)

In [0]:
# List of datasets from GDP
load_df = [
'party_case_client_details',
'party_AllPartyDetails' 
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService',Load_Date=Load_Date)

In [0]:
import pandas as pd # print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'pessoa'
,'vwgic_rdl_pessoa_juridica_identificacao'
,'tipo_sociedade'
,'vwgic_rdl_pessoa'
]})

# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GIC', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:

# print list of strings for loading spark dfs from GDP
load_df = pd.DataFrame({'GDPname':[
'ADKYC_CONTRAPARTES'
,'ADKYC_CONTRAPARTES_COMPL'
,'ADRBB_CONTRAPARTES_QUESTOES_RESPOSTAS'
,'ADKYC_SITUACOES'
,'ADRBB_CONTRAPARTES_GRAM'
,'ADKYC_RISCOS'
,'ADRBB_CONTRAPARTES_PONTUACOES_ABAS'
,'ADRBB_DOMINIOS_QUESTOES'
,'ADRBB_QUESTOES'
]})

# Create TempView for each loading table
for index, row in load_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='KN1', Dataobject=row.GDPname,Load_Date=Load_Date)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df = [
'Legacy2_case_client_details'
, 'Legacy2_risk'
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='Legacy2', Dataobject=Object,path_prefix='Legacy2',Load_Date=Load_Date)

In [0]:
df_Party_SystemIdentifier=spark.read.parquet(f'abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_SystemIdentifier/3/data/{load_dts}/*.parquet')
df_Party_KN1Cases=spark.read.parquet(f'abfss://radardatamodel@{SARADAR}.dfs.core.windows.net/Party_KN1Cases/3/data/{load_dts}/*.parquet')
df_Party_KN1Cases.createOrReplaceTempView('KN1Cases')

In [0]:
load_df = [
    'c_b_party_xref'
,'c_lkp_cdd_risk_rating_xref'
,'c_b_due_diligence_xref'
,'c_b_contract_xref'
,'c_b_contr_rol_party_xref'
]
load_ranz_v4_rdm_tables(load_df, Load_Date)

In [0]:
# import pandas as pd # print list of strings for loading spark dfs from GDP
# load_df = pd.DataFrame({'GDPname':[
# 'nls_dbo_cif'
# ,'nls_dbo_loanacct'
# ,'nls_dbo_loanacct_detail'
# ]})
# #gic_load_dts
# # Create TempView for each loading table
# for index, row in load_df.iterrows():
#     Read_GDP_Defined_DataObjects(Source='NLSVF', Dataobject=row.GDPname)

In [0]:
#Creating Static view for Risk Level
Risk_list = [0,1,2,3,4,5,6,7]
Risk_Description = ['Undefined', 'No', 'Low', 'Medium', 'High','Super','Extra High','Unacceptable']
#Creating spark dataframe from lists
df_gcob_static_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['Id', 'Description'])
df_gcob_static_RiskLevel.createOrReplaceTempView('static_RiskLevel')

#Creating Static view for Risk Level
Risk_list = [0,1,2,3,4,9999]
Risk_Description = ['RiskNotAssigned','Unacceptable', 'High','Medium', 'Low', 'Unknown']
df_KN1_static_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['Id', 'Description'])
df_KN1_static_RiskLevel.createOrReplaceTempView('kn1_static_RiskLevel')

#Creating Static view for KN1 Risk Level
Risk_list = [0,1,2,3,4]
Risk_Description = ['Incomplete', 'Low', 'Medium', 'High','Unacceptable']
#Creating spark dataframe from lists
df_gcob_RiskLevel = spark.createDataFrame(zip(Risk_list, Risk_Description), ['RiskModelRiskLevelId', 'DisplayName'])
df_gcob_RiskLevel.createOrReplaceTempView('RiskModelRiskLevel')


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW RiskModelCategories AS
with InstanceCategory as (
  select
    InstanceId,
    Geographical AS GeoRisk,
    `Entity Type` AS EntityTypeRisk,
    Structure AS StructureRisk,
    Sector AS SectorRisk,
    `Products and Services` AS ProductRisk,
    PEP AS PEPRisk,
    `Transaction` AS TransactionRisk,
    `Distribution Channel` AS DistributionRisk,
    `Third Party` AS ThirdPartyRisk,
    `Adverse Info` AS AdverseInfoRisk,
    Other AS OtherRisk
  FROM
    (
      select
        InstanceId,
        MAX(
          CASE
            WHEN CategoryName = 'Adverse Info' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Adverse Info`,
        MAX(
          CASE
            WHEN CategoryName = 'Distribution Channel' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Distribution Channel`,
        MAX(
          CASE
            WHEN CategoryName = 'Entity Type' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Entity Type`,
        MAX(
          CASE
            WHEN CategoryName = 'General' then CalculatedRiskLevelId
            else NULL
          END
        ) as General,
        MAX(
          CASE
            WHEN CategoryName = 'Geographical' then CalculatedRiskLevelId
            else NULL
          END
        ) as Geographical,
        MAX(
          CASE
            WHEN CategoryName = 'Other' then CalculatedRiskLevelId
            else NULL
          END
        ) as Other,
        MAX(
          CASE
            WHEN CategoryName = 'PEP' then CalculatedRiskLevelId
            else NULL
          END
        ) as PEP,
        MAX(
          CASE
            WHEN CategoryName = 'Products and Services' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Products and Services`,
        MAX(
          CASE
            WHEN CategoryName = 'Sector' then CalculatedRiskLevelId
            else NULL
          END
        ) as Sector,
        MAX(
          CASE
            WHEN CategoryName = 'Structure' then CalculatedRiskLevelId
            else NULL
          END
        ) as Structure,
        MAX(
          CASE
            WHEN CategoryName = 'Third Party' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Third Party`,
        MAX(
          CASE
            WHEN CategoryName = 'Transaction' then CalculatedRiskLevelId
            else NULL
          END
        ) as `Transaction`
      FROM
        Party_RiskModelCategories
      GROUP BY
        InstanceId
    )
)
select distinct
  geo.Description as GeographicalRiskLevel,
  Ent.Description as EntityTypeRiskLevel,
  stru.Description as StructureRiskLevel,
  sec.Description as SectorRiskLevel,
  prod.Description as ProductAndServiceRiskLevel,
  pep.Description as PEPRiskLevel,
  tran.Description as TransactionRiskLevel,
  dist.Description as DistributionRiskLevel,
  thir.Description as ThirdPartyRiskLevel,
  adv.Description as AdverseInfoRiskLevel,
  oth.Description as OtherRiskLevel,
  InstanceId
FROM
  InstanceCategory AS rf
    LEFT OUTER JOIN static_RiskLevel geo
      ON rf.GeoRisk = geo.Id
    LEFT OUTER JOIN static_RiskLevel Ent
      ON rf.EntityTypeRisk = Ent.Id
    LEFT OUTER JOIN static_RiskLevel stru
      ON rf.StructureRisk = stru.Id
    LEFT OUTER JOIN static_RiskLevel sec
      ON rf.SectorRisk = sec.Id
    LEFT OUTER JOIN static_RiskLevel prod
      ON rf.ProductRisk = prod.Id
    LEFT OUTER JOIN static_RiskLevel pep
      ON rf.PEPRisk = pep.Id
    LEFT OUTER JOIN static_RiskLevel tran
      ON rf.TransactionRisk = tran.Id
    LEFT OUTER JOIN static_RiskLevel dist
      ON rf.DistributionRisk = dist.Id
    LEFT OUTER JOIN static_RiskLevel thir
      ON rf.ThirdPartyRisk = thir.Id
    LEFT OUTER JOIN static_RiskLevel adv
      ON rf.AdverseInfoRisk = adv.Id
    LEFT OUTER JOIN static_RiskLevel oth
      ON rf.otherRisk = oth.Id

In [0]:
%sql
Create or replace temporary view Gcob_CaseClientDetails As
select
  *,
  case
    when ClientType = 'Legal Entity' then 'Legal Entity'
    when
      ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)')
    then
      'Natural Person'
    else null
  end as Party_type,
  Case
    when ClientType = 'Legal Entity' then concat('LEC_', GcobId)
    when
      ClientType in ('Natural Person', 'Natural Person acting in a Professional Capacity (NPPC)')
    then
      concat('NP_NPPC_', GcobId)
  End as UniquePartyId
from
  party_case_client_details
where
  CaseStatusName <> 'Cancelled'

In [0]:
%sql
Create or replace temporary view Gcob_ApplicableRisk As
select distinct
SourceClient
,GeographicalApplicableRisk as GeographicalIsMaterial
,EntityTypeApplicableRisk as EntityTypeIsMaterial
,StructureApplicableRisk as StructureIsMaterial
,SectorApplicableRisk as SectorIsMaterial
,ProductsApplicableRisk as ProductsAndServicesIsMaterial
,PoliticallyExposedPersonsApplicableRisk as PEPIsMaterial
,TransactionApplicableRisk as TransactionIsMaterial
,DistributionChannelApplicableRisk as DistributionChannelIsMaterial
,ThirdPartyApplicableRisk as ThirdPartyIsMaterial
,AdverseInfoApplicableRisk as AdverseInfoIsMaterial
,OtherApplicableRisk as OtherIsMaterial
from Gcob_CaseClientDetails

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW GCOB_MaterialRisk AS
with QuestionLevelMateriality_material as 
(
select distinct
qa.InstanceId
,case when qa.MaterialQuestionCode like 'GEO%' then 'Geographical'
      when qa.MaterialQuestionCode like 'STR%' then 'Structure'
      when qa.MaterialQuestionCode like 'SEC%' then 'Sector'
      when qa.MaterialQuestionCode like 'TX%' then 'Transaction'
      when qa.MaterialQuestionCode like 'PS%' then 'Products and Services'
      when qa.MaterialQuestionCode like 'ADVR%' then 'Adverse Info'
      when qa.MaterialQuestionCode like 'ENT%' then 'Entity Type'
      when qa.MaterialQuestionCode like 'PEP%' then 'PEP'
      when qa.MaterialQuestionCode like '3RD%' then 'Third Party'
      when qa.MaterialQuestionCode like 'DIST%' then 'Distribution Channel'
      when qa.MaterialQuestionCode like 'OTH%' then 'Other'
  end as CategoryName
,qa.IsMaterial
,qa.ClientId,qa.SourceClient,qa.CaseStatusName,qa.IsLatestApprovedVersionOfClient
from Party_RiskModelInstanceQuestionAnswers qa
where qa.SourceSystemName = 'GCOB-CaseService' and qa.ClientId is not null and qa.ModelType = 'QuestionLevelMateriality'
)
,QuestionLevelMateriality AS
(
select distinct
InstanceId,SourceClient,
MAX(CASE WHEN CategoryName = 'Adverse Info' then IsMaterial else NULL END ) as AdverseInfoIsMaterial,
MAX(CASE WHEN CategoryName = 'Distribution Channel' then IsMaterial else NULL END ) as DistributionChannelIsMaterial,
MAX(CASE WHEN CategoryName = 'Entity Type' then IsMaterial else NULL END ) as EntityTypeIsMaterial,
MAX(CASE WHEN CategoryName = 'Geographical' then IsMaterial else NULL END ) as GeographicalIsMaterial,
MAX(CASE WHEN CategoryName = 'Other' then IsMaterial else NULL END ) as OtherIsMaterial,
MAX(CASE WHEN CategoryName = 'PEP' then IsMaterial else NULL END ) as PEPIsMaterial,
MAX(CASE WHEN CategoryName = 'Products and Services' then IsMaterial else NULL END ) as ProductsAndServicesIsMaterial,
MAX(CASE WHEN CategoryName = 'Sector' then IsMaterial else NULL END ) as SectorIsMaterial,
MAX(CASE WHEN CategoryName = 'Structure' then IsMaterial else NULL END ) as StructureIsMaterial,
MAX(CASE WHEN CategoryName = 'Third Party' then IsMaterial else NULL END ) as ThirdPartyIsMaterial,
MAX(CASE WHEN CategoryName = 'Transaction' then IsMaterial else NULL END ) as TransactionIsMaterial
FROM QuestionLevelMateriality_material
GROUP BY InstanceId,SourceClient
)
,defaultModel as
(
select distinct 
d.InstanceId,
a.SourceClient,
cast(a.AdverseInfoIsMaterial as BOOLEAN) as AdverseInfoIsMaterial,
cast(a.DistributionChannelIsMaterial as BOOLEAN) as DistributionChannelIsMaterial,
cast(a.EntityTypeIsMaterial as BOOLEAN) as EntityTypeIsMaterial,
cast(a.GeographicalIsMaterial as BOOLEAN) as GeographicalIsMaterial,
cast(a.OtherIsMaterial as BOOLEAN) as OtherIsMaterial,
cast(a.PEPIsMaterial as BOOLEAN) as PEPIsMaterial,
cast(a.ProductsAndServicesIsMaterial as BOOLEAN) as ProductsAndServicesIsMaterial,
cast(a.SectorIsMaterial as BOOLEAN) as SectorIsMaterial,
cast(a.StructureIsMaterial as BOOLEAN) as StructureIsMaterial,
cast(a.ThirdPartyIsMaterial as BOOLEAN) as ThirdPartyIsMaterial,
cast(a.TransactionIsMaterial as BOOLEAN) as TransactionIsMaterial
from Party_RiskModelInstanceQuestionAnswers d
inner join Gcob_ApplicableRisk a on d.SourceClient = a.SourceClient
where d.SourceSystemName = 'GCOB-CaseService' and d.ClientId is not null and d.ModelType = 'Default'
)
select * from QuestionLevelMateriality
union
select * from defaultModel

In [0]:
%sql
Create or replace temporary view KN1_Dim_Case As
with KN1_Fianl_cases as 
(
SELECT distinct
'KN1' AS SourceSystem
, GicID AS ClientId
, PartyLifeCycleStatus2 AS ClientLifeCycleStatus
, CaseID AS CaseId
, NULL AS CaseClientID
, ReviewType
, NULL AS IsFullReview
, CASE WHEN b.CLientID IS NULL  THEN 0 ELSE 1 END AS IsLatestID
, CaseCreationDate
, CaseStatus
, CaseCompletedDate
, CaseEndDate
, NULL AS FinalDecisionDate_From
, NULL AS FinalDecisionDate_To
, NULL AS ClientApprovalDate_From
, NULL AS ClientApprovalDate_To
, NULL AS Expired
, NextReviewDate
FROM KN1Cases a
LEFT JOIN ( 
SELECT 
GicID AS ClientID
, Max(CaseCompletedDate) AS LatestDate
FROM KN1Cases 
WHERE
(CaseStatus != 'Cancelled' and PartyLifeCycleStatus2 = 'Client')
GROUP BY GicID
 ) b 
ON a.CaseCompletedDate = b.LatestDate 
AND b.ClientID=a.GicID

WHERE 
CaseCompletedDate <= COALESCE(OffboardingDate_PartyRole, '2199-12-31')
)
select distinct 
  cases.SourceSystem
, cases.Clientid
, cases.ClientLifeCycleStatus 
, cases.CaseId
, cases.CaseClientID
, cases.ReviewType as CaseReviewType
, cases.IsFullReview
, cases.IsLatestID--0 or 1
, cases.CaseCreationDate as CaseCreatedDate
, cases.CaseStatus as CurrentCasestatus
, case
wHEN cases.CaseStatus = 'Cancelled' THEN 'Cancelled'
wHEN cases.CaseStatus in ('Offboarding','Approved','Completed','Reapproved','Reactivated') THEN 'Completed'
ELSE 'In Progress' END AS StandardCaseStatus
, cases.CaseCompletedDate as DateCompleted_From
, cases.CaseEndDate as DateCompleted_To
, cases.FinalDecisionDate_From
, cases.FinalDecisionDate_To
, cases.ClientApprovalDate_From
, cases.ClientApprovalDate_To
, cases.Expired
, cases.NextReviewDate
from KN1_Fianl_cases as cases

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_GRAM_Casesdetails AS
select distinct
  CONCAT('GIC_',c.ClientId) as LocalSystemIdentifier,
  c.ClientId,
  kn_cont.ID_CONTRAPARTE as CaseID,
  r.Description as CalculatedRiskLevel,
  rl.Description as RecalculatedRiskLevel,
  gram.InstanceId,
  gram.MaterialInstanceId,
  i.Status as GRAM_status
from
  ADKYC_CONTRAPARTES kn_cont
  Inner join ADRBB_CONTRAPARTES_GRAM stat on trim(kn_cont.ID_CONTRAPARTE) = trim(stat.ID_CONTRAPARTE)
  INNER JOIN KN1_Dim_Case c on trim(kn_cont.ID_CONTRAPARTE) = trim(c.CaseId)
  INNER JOIN Party_RiskModelInstanceQuestionAnswers gram on trim(stat.ID_GRAM_IDENTITY) = trim(gram.InstanceId)
    and trim(gram.SourceSystemName) = 'Brazil - KN1'
  Left Join Party_RiskModelInstance i on gram.InstanceId = i.InstanceId
  left join kn1_static_RiskLevel r on trim(kn_cont.CD_RISCO_MANUAL) = trim(r.Id)
  left join kn1_static_RiskLevel rl on trim(kn_cont.CD_RISCO_CALCULADO) = trim(rl.Id)
  

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_GRAM_Risk AS
select distinct
  knc.LocalSystemIdentifier,
  knc.ClientID,
  knc.CaseID,
  c.GeographicalRiskLevel,
  c.EntityTypeRiskLevel,
  c.StructureRiskLevel,
  c.SectorRiskLevel,
  c.ProductAndServiceRiskLevel,
  c.PEPRiskLevel,
  c.TransactionRiskLevel,
  c.DistributionRiskLevel,
  c.ThirdPartyRiskLevel,
  c.AdverseInfoRiskLevel,
  knc.CalculatedRiskLevel,
  knc.RecalculatedRiskLevel,
  m.GeographicalIsMaterial as GeographicalMaterialRisk,
  m.EntityTypeIsMaterial as EntityTypeMaterialRisk,
  m.StructureIsMaterial as StructureMaterialRisk,
  m.SectorIsMaterial as SectorMaterialRisk,
  m.ProductsAndServicesIsMaterial as ProductAndServiceMatrialRisk,
  m.PEPIsMaterial as PEPMaterialRisk,
  m.TransactionIsMaterial as TransactionMaterialRisk,
  m.DistributionChannelIsMaterial as DistributionMatrialRisk,
  m.ThirdPartyIsMaterial as ThirdPartyMaterialRisk,
  m.AdverseInfoIsMaterial as AdverseInfoMaterialRisk,
  knc.InstanceId as GRAM_InstanceID,
  knc.GRAM_status
from
KN1_GRAM_Casesdetails knc
left join RiskModelCategories c on c.InstanceId = knc.InstanceId
left join GCOB_MaterialRisk m on m.InstanceId = knc.MaterialInstanceId

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_GRAM_DerivedRisk AS
with cte  as (
		 SELECT 'Baixo' as Portugese, 'Low' as English UNION
		 SELECT 'Médio', 'Medium' UNION
		 SELECT 'Alto','High' UNION
		 SELECT 'Novo', 'Onboarding' UNION
		 SELECT 'Renovação', 'Renewal' UNION
		 SELECT 'Aprovado', 'Approved' UNION
		 SELECT 'Reativado', 'Reactivated' UNION
		 SELECT 'Ativo', 'Client' UNION 
		 SELECT 'Bloqueado', 'Blocked' UNION
		 SELECT 'Reativação', 'Client' UNION--'Reactivated' 
		 SELECT 'Inativo', 'Former Client' UNION
		 SELECT 'Pré Cadastro', 'Prospect' UNION
		 SELECT 'Risco Não Atribuído', 'Unassigned'
	 )

SELECT distinct
DC.SourceSystem
, DC.ClientId
, DC.PartyLifeCycleStatus
, DC.CurrentCasestatus
, DC.CaseCreatedDate
, DC.NextReviewDate
, DC.DateCompleted_From
, DC.DateCompleted_To
, RISF.English AS ValidatedRiskDescription
, RISF.riskmodelrisklevelid AS ValidatedRisk
, legalent.DES_TIPO_SOCIEDADE as GIC_entitytype
, CASE WHEN person.SEQ_TIPO_PESSOA = '2' THEN 'Yes' ELSE 'No' END AS isNaturalPerson
, SEQ_TIPO_SOCIEDADE
, B.*

FROM (
SELECT SourceSystem
, ClientId
, ClientLifeCycleStatus AS PartyLifeCycleStatus
, CaseId
, CurrentCasestatus
, CaseCreatedDate
, NextReviewDate
, DateCompleted_From
, DateCompleted_To
FROM KN1_Dim_Case
WHERE sourcesystem = 'KN1'
AND StandardCaseStatus = 'Completed'
) DC

LEFT JOIN (
SELECT ID_CONTRAPARTE
, CD_RISCO_ATRIBUIDO
FROM ADKYC_CONTRAPARTES
) AS COPA
ON COPA.ID_CONTRAPARTE = DC.CaseId
LEFT JOIN (
SELECT NM_RISCO
, CD_RISCO
, cte.ENGLISH
, rl.riskmodelrisklevelid
, rl.DisplayName
FROM ADKYC_RISCOS
LEFT JOIN CTE cte
ON NM_RISCO = cte.portugese
LEFT JOIN RiskModelRiskLevel rl
ON replace(rl.DisplayName, ' risk', '') = cte.English
) RISF
ON COPA.CD_RISCO_ATRIBUIDO = RISF.CD_RISCO
LEFT JOIN (
SELECT COD_INSTITUCIONAL
, ident.SEQ_TIPO_SOCIEDADE
, DES_TIPO_SOCIEDADE
FROM vwgic_rdl_pessoa_juridica_identificacao ident
LEFT JOIN (
SELECT SEQ_TIPO_SOCIEDADE
,DES_TIPO_SOCIEDADE
FROM tipo_sociedade
) soc
ON soc.SEQ_TIPO_SOCIEDADE = ident.SEQ_TIPO_SOCIEDADE
) legalent
ON DC.ClientId = legalent.COD_INSTITUCIONAL
LEFT JOIN (
SELECT 
COD_INSTITUCIONAL
,SEQ_TIPO_PESSOA
FROM vwgic_rdl_pessoa
) person
ON DC.ClientId= person.COD_INSTITUCIONAL

LEFT JOIN (
 select 
 ROW_NUMBER() OVER (PARTITION BY CaseId ORDER BY CASE WHEN GRAM_status IN ('Complete', 'CompleteLocked') THEN 0 ELSE 1 END
				,CASE WHEN risksourcesystem = 'KN1' THEN 1 WHEN risksourcesystem = 'GRAM' THEN 2 END
			) as RN
 ,C.*
	FROM (

SELECT
'KN1' AS risksourcesystem
, dc.CaseId
, NULL AS GRAM_InstanceID
, NULL AS GRAM_status
, null as CalculatedRisk
, RISC.name AS CalculatedRiskDescription
, null as ReCalculatedRisk
, RISI.name AS RecalculatedRiskDescription
, RADV.ID AS AdverseInfoRisk
, RADV.DE_PONTUACAO_ABA_EN AS AdverseInfoRiskDescription
, RDIS.ID AS DistributionRisk
, RDIS.DE_PONTUACAO_ABA_EN AS DistributionRiskDescription
, RENT.ID AS EntityRisk
, RENT.DE_PONTUACAO_ABA_EN AS EntityRiskDescription
, RGEO.ID AS GeoRisk
, RGEO.DE_PONTUACAO_ABA_EN AS GeoRiskDescription
, NULL AS OtherRisk
, NULL AS OtherRiskDescription
, RPEP.ID AS PEPRisk
, RPEP.DE_PONTUACAO_ABA_EN AS PEPRiskDescription
, RPRO.ID AS ProductRisk
, RPRO.DE_PONTUACAO_ABA_EN AS ProductRiskDescription
, RSEC.ID AS SectorRisk
, RSEC.DE_PONTUACAO_ABA_EN AS SectorRiskDescription
, RSTR.ID AS StructureRisk
, RSTR.DE_PONTUACAO_ABA_EN AS StructureRiskDescription
, RTHP.ID AS ThirdPartyRisk
, RTHP.DE_PONTUACAO_ABA_EN AS ThirdPartyRiskDescription
, RTRA.ID AS TransactionRisk
, RTRA.DE_PONTUACAO_ABA_EN AS TransactionRiskDescription
, NULL AS AdverseInfoIsMaterial
, NULL AS DistributionChannelIsMaterial
, NULL AS EntityTypeIsMaterial
, NULL AS GeographicalIsMaterial
, NULL AS OtherIsMaterial
, NULL AS PEPIsMaterial
, NULL AS ProductsAndServicesIsMaterial
, NULL AS SectorIsMaterial
, NULL AS StructureIsMaterial
, NULL AS ThirdPartyIsMaterial
, NULL AS TransactionIsMaterial
, NULL AS STR4Layers
, NULL AS StructureTax
, IsTrust as IsTrust
, TCSP AS StructureTCSP
, ComplexStructure AS StructureOverlyComplex
, FaceToFaceContact AS `Face2Face Contact`
, adverseinfo AS AdverseInfo
, IssuesBearerShares AS IssuesBearerShares
, NomineeShareholders AS NomineeShareholders
, entityType AS EntityType 
from (
SELECT SourceSystem
, ClientId
, CaseId
, DateCompleted_From
, DateCompleted_To
FROM KN1_Dim_Case
WHERE
sourcesystem = 'KN1'
AND StandardCaseStatus = 'Completed'
) dc
LEFT JOIN (
SELECT ID_CONTRAPARTE
, CD_RISCO_CALCULADO
, CD_RISCO_MANUAL
FROM ADKYC_CONTRAPARTES
) AS COPA
ON COPA.ID_CONTRAPARTE = DC.CaseId
LEFT JOIN (
SELECT NM_RISCO
, CD_RISCO
, rl.id
, rl.Description as name
FROM ADKYC_RISCOS
left join kn1_static_RiskLevel rl on CD_RISCO = rl.Id
-- LEFT JOIN CTE cte
-- ON NM_RISCO = cte.portugese
-- LEFT JOIN static_RiskLevel rl
-- ON replace(rl.Description, ' risk', '') = cte.English
) RISC
ON COPA.CD_RISCO_MANUAL = RISC.CD_RISCO
LEFT JOIN (
SELECT NM_RISCO
, CD_RISCO
, rl.id
, rl.Description as name
FROM ADKYC_RISCOS
left join kn1_static_RiskLevel rl on CD_RISCO = rl.Id
-- LEFT JOIN CTE cte
-- ON NM_RISCO = cte.portugese
-- LEFT JOIN static_RiskLevel rl
-- ON replace(rl.Description, ' risk', '') = cte.English
) RISI
ON COPA.CD_RISCO_CALCULADO = RISI.CD_RISCO
LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Geográfico'
) RGEO
ON RGEO.ID_CONTRAPARTE = dc.CaseId
LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Tipos de Empresas'
) RENT
ON RENT.ID_CONTRAPARTE = dc.CaseId
LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Estrutural'
) RSTR
ON RSTR.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Setores'
) RSEC
ON RSEC.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Produtos'
) RPRO
ON RPRO.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'PEP'
) RPEP
ON RPEP.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Transacional'
) RTRA
ON RTRA.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Canal de Distribuição'
) RDIS
ON RDIS.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Terceiros'
) RTHP
ON RTHP.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT ABAS.ID_CONTRAPARTE
, ABAS.DE_PONTUACAO_ABA
, DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO
, cte.English AS DE_PONTUACAO_ABA_EN
, rl.ID
FROM ADRBB_CONTRAPARTES_PONTUACOES_ABAS ABAS
LEFT JOIN ADRBB_DOMINIOS_QUESTOES DOMIN
ON DOMIN.CD_DOMINIO_QUESTAO = ABAS.CD_DOMINIO_QUESTAO
LEFT JOIN CTE cte
ON ABAS.DE_PONTUACAO_ABA = cte.portugese
LEFT JOIN static_RiskLevel rl
ON replace(rl.Description, ' risk', '') = cte.English
WHERE DOMIN.DE_SUB_ABA_DOMINIO_QUESTAO = 'Informação Adversa'
) RADV
ON RADV.ID_CONTRAPARTE = dc.CaseId

LEFT JOIN (
SELECT distinct
ID_CONTRAPARTE, 
MAX(CASE WHEN cat = 'ComplexStructure' THEN Answer END) AS ComplexStructure,
MAX(CASE WHEN cat = 'NomineeShareholders' THEN Answer END) AS NomineeShareholders,
MAX(CASE WHEN cat = 'IssuesBearerShares' THEN Answer END) AS IssuesBearerShares,
MAX(CASE WHEN cat = 'IsTrust' THEN Answer END) AS IsTrust,
MAX(CASE WHEN cat = 'TCSP' THEN Answer END) AS TCSP,
MAX(CASE WHEN cat = 'FaceToFaceContact' THEN Answer END) AS FaceToFaceContact,
MAX(CASE WHEN cat = 'adverseinfo' THEN Answer END) AS adverseinfo,
MAX(CASE WHEN cat = 'entityType' THEN Answer END) AS entityType
FROM (
SELECT ID_CONTRAPARTE
, cat
, Answer
FROM (
SELECT ROW_NUMBER() OVER (PARTITION BY cppc.ID_CONTRAPARTE, Q_cat.cat ORDER BY CPPC.CD_RESPOSTA DESC) as RN
, cppc.CD_RESPOSTA
, cppc.ID_QUESTAO
, cppc.ID_CONTRAPARTE
, doqu.CD_QUESTAO
, Q_cat.cat
, CASE 
WHEN doqu.CD_QUESTAO = 'Q47' AND CPPC.CD_RESPOSTA = 2 THEN 'Yes'
WHEN doqu.CD_QUESTAO = 'Q47' AND CPPC.CD_RESPOSTA != 2 THEN NULL
when tl.EN is not null then tl.EN else CD_RESPOSTA END AS Answer
FROM ADRBB_CONTRAPARTES_QUESTOES_RESPOSTAS CPPC
INNER JOIN (
SELECT ID_QUESTAO
, CD_QUESTAO 
FROM ADRBB_QUESTOES
) DOQU
ON DOQU.ID_QUESTAO = CPPC.ID_QUESTAO
INNER JOIN (
SELECT 'Q286' AS CD_QUESTAO
, 'ComplexStructure' AS cat

UNION

SELECT 'Q292', 'NomineeShareholders'

UNION 

SELECT 'Q314', 'IssuesBearerShares'

UNION

SELECT 'Q311', 'IsTrust'

UNION

SELECT 'Q287', 'TCSP'

UNION 

SELECT 'Q42', 'FaceToFaceContact'

UNION 

SELECT 'Q77', 'FaceToFaceContact'

UNION

SELECT 'Q47', 'adverseinfo' 

UNION 

SELECT 'Q5', 'entityType'

) Q_cat
ON Q_cat.CD_QUESTAO = DOQU.CD_QUESTAO
LEFT JOIN (
SELECT 'Não' AS BR, 'No' AS EN
UNION
SELECT 'Sim' AS BR, 'Yes' AS EN
) tl
ON tl.BR = case when DOQU.CD_QUESTAO in ('Q311', 'Q312', 'Q314') then de_resposta else cppc.CD_RESPOSTA end
) a
) b
GROUP BY ID_CONTRAPARTE
) charact
ON charact.ID_CONTRAPARTE = dc.CaseId

Union

select distinct 
'GRAM' AS risksourcesystem
,CaseID
,GRAM_InstanceID
,GRAM_status as GRAM_status
,null as id_CalculatedRisk
,CalculatedRiskLevel
,null as id_ReCalculatedRisk
,RecalculatedRiskLevel
,null as id_AdverseInfoRiskLevel
,AdverseInfoRiskLevel
,null as id_DistributionRiskLevel
,DistributionRiskLevel
,null as id_EntityTypeRiskLevel
,EntityTypeRiskLevel
,null as id_GeographicalRiskLevel
,GeographicalRiskLevel
,null as id_OtherRiskLevel
,null as OtherRiskLevel
,null as id_PEPRiskLevel
,PEPRiskLevel
,null as id_ProductAndServiceRiskLevel
,ProductAndServiceRiskLevel
,null as id_SectorRiskLevel
,SectorRiskLevel
,null as id_StructureRiskLevel
,StructureRiskLevel
,null as id_ThirdPartyRiskLevel
,ThirdPartyRiskLevel
,null as id_TransactionRiskLevel
,TransactionRiskLevel
,AdverseInfoMaterialRisk
,DistributionMatrialRisk
,EntityTypeMaterialRisk
,GeographicalMaterialRisk
,null as OtherIsMaterial
,PEPMaterialRisk
,ProductAndServiceMatrialRisk
,SectorMaterialRisk
,StructureMaterialRisk
,ThirdPartyMaterialRisk
,TransactionMaterialRisk
, null as STR4Layers
, null as StructureTax
, null as IsTrust
, null as StructureTCSP
, null as StructureOverlyComplex
, null as Face2FaceContact
, null as AdverseInfo
, null as IssuesBearerShares
, null as NomineeShareholders
, null as EntityType
from KN1_GRAM_Risk
) C
) B
ON B.CaseId = DC.CaseId
WHERE RN = 1


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW KN1_Risk AS
select distinct
CASE 
  WHEN TRIM(ClientId) = '' THEN CONCAT('KN1_',CaseId) 
  ELSE CONCAT('GIC_',ClientId) 
END as LocalSystemIdentifier, -- PBI 14525246
CASE 
  WHEN TRIM(ClientId) = '' THEN NULL
  ELSE ClientId 
END as ClientId -- PBI 14525246
,CaseId
,case when isNaturalPerson = 'No' then concat('KN1_LEC_', CaseId) ELSE concat('KN1_NP_NPPC_', CaseId) END AS UniqueCaseId
,GeoRiskDescription as GeographicalRiskLevel
,EntityRiskDescription as EntityTypeRiskLevel
,StructureRiskDescription as StructureRiskLevel
,SectorRiskDescription as SectorRiskLevel
,ProductRiskDescription as ProductAndServiceRiskLevel
,PEPRiskDescription as PEPRiskLevel
,TransactionRiskDescription as TransactionRiskLevel
,DistributionRiskDescription as DistributionRiskLevel
,ThirdPartyRiskDescription as ThirdPartyRiskLevel
,AdverseInfoRiskDescription as AdverseInfoRiskLevel
,CalculatedRiskDescription as CalculatedRiskLevel
,RecalculatedRiskDescription as RecalculatedRiskLevel
,OtherRiskDescription as OtherRiskLevel
,AdverseInfoIsMaterial
,DistributionChannelIsMaterial
,EntityTypeIsMaterial
,GeographicalIsMaterial
,PEPIsMaterial
,ProductsAndServicesIsMaterial
,SectorIsMaterial
,StructureIsMaterial
,ThirdPartyIsMaterial
,TransactionIsMaterial
,OtherIsMaterial
from KN1_GRAM_DerivedRisk

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gcob_casesrisk AS
select distinct
  CONCAT('GCOB_',c.UniquePartyId) AS LocalSystemIdentifier,
  c.ClientId,
  c.CaseId,
  CASE
    WHEN c.ClientType like 'Legal%' THEN concat(c.SourceSystem, '_LEC_', c.CaseId)
    WHEN c.ClientType LIKE 'Natural%' THEN concat(c.SourceSystem, '_NP_NPPC_', c.CaseId)
  END as UniqueCaseId,
  c.GeographicalRiskLevel,
  c.EntityTypeRiskLevel,
  c.StructureRiskLevel,
  c.SectorRiskLevel,
  c.ProductAndServiceRiskLevel,
  c.PEPRiskLevel,
  c.TransactionRiskLevel,
  c.DistributionRiskLevel,
  c.ThirdPartyRiskLevel,
  c.AdverseInfoRiskLevel,
  c.ModelCalculatedRiskLevel as CalculatedRiskLevel,
  c.ModelRecalculatedRiskLevel as RecalculatedRiskLevel
  ,c.OtherRiskLevel
  ,m.AdverseInfoIsMaterial
  ,m.DistributionChannelIsMaterial
  ,m.EntityTypeIsMaterial
  ,m.GeographicalIsMaterial
  ,m.PEPIsMaterial
  ,m.ProductsAndServicesIsMaterial
  ,m.SectorIsMaterial
  ,m.StructureIsMaterial
  ,m.ThirdPartyIsMaterial
  ,m.TransactionIsMaterial
  ,null as OtherIsMaterial
from Gcob_CaseClientDetails c
left join GCOB_MaterialRisk m on c.SourceClient = m.SourceClient
where
  c.CaseStatusName <> 'Cancelled'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW NonSFDCID_Legacy2_Client AS
select
  *
from
  Legacy2_case_client_details
where
  isclient = 1
  and GcobCaseId is null
  and Value is null
  and GcobId like 'RA:%';

CREATE OR REPLACE TEMPORARY VIEW Legacy2_client AS
select
  *
from
  Legacy2_case_client_details
where
  ClientId not in (
    select
      ClientId
    from
      NonSFDCID_Legacy2_Client
  );

CREATE OR REPLACE TEMPORARY VIEW Legacy2_riskdetails AS
select distinct
  CONCAT('LEGACY2_', CASE
    WHEN IsClient = 'true'
    and ClientTypeId = 1 THEN concat('LE_', c.GcobId)
    WHEN IsClient = 'true'
    and ClientTypeId in (2, 3) THEN concat('NP_NPPC_', c.GcobId)
    WHEN IsClient = 'false'
    and ClientTypeId = 1 THEN concat('RLE_', c.GcobId)
    WHEN IsClient = 'false'
    and ClientTypeId in (2, 3) THEN concat('RNP_', c.GcobId) END) as LocalSystemIdentifier,
  c.ClientId,
  null as CaseId,
  CASE
    WHEN c.IsClient = 'true'
    and c.ClientTypeId = 1 THEN concat(c.SourceSystem, '_LEC_', c.ClientId)
    WHEN c.IsClient = 'true'
    and c.ClientTypeId in (2, 3) THEN concat(c.SourceSystem, '_NP_NPPC_', c.ClientId)
    WHEN c.IsClient = 'false'
    and c.ClientTypeId = 1 THEN concat(c.SourceSystem, '_RLEP_', c.ClientId)
    WHEN c.IsClient = 'false'
    and c.ClientTypeId in (2, 3) THEN concat(c.SourceSystem, '_RNPP_', c.ClientId)
  END as UniqueCaseId,
  r.GeographicalRetainedRiskLevel as GeographicalRiskLevel,
  r.EntityTypeRetainedRiskLevel as EntityTypeRiskLevel,
  r.StructureRetainedRiskLevel as StructureRiskLevel,
  r.SectorRetainedRiskLevel as SectorRiskLevel,
  r.ProductsRetainedRiskLevel as ProductAndServiceRiskLevel,
  r.PoliticallyExposedPersonsRetainedRiskLevel as PEPRiskLevel,
  r.TransactionRetainedRiskLevel as TransactionRiskLevel,
  r.DistributionChannelRetainedRiskLevel as DistributionRiskLevel,
  r.ThirdPartyRetainedRiskLevel as ThirdPartyRiskLevel,
  r.AdverseInfoRetainedRiskLevel as AdverseInfoRiskLevel,
  r.CalculatedRiskLevel,
  r.RecalculatedRiskLevel
  ,null as OtherRiskLevel
  ,cast(r.AdverseInfoApplicableRisk as BOOLEAN) as AdverseInfoIsMaterial
  ,cast(r.DistributionChannelApplicableRisk as BOOLEAN) as DistributionChannelIsMaterial
  ,cast(r.EntityTypeApplicableRisk as BOOLEAN) as EntityTypeIsMaterial
  ,cast(r.GeographicalApplicableRisk as BOOLEAN) as GeographicalIsMaterial
  ,cast(r.PoliticallyExposedPersonsApplicableRisk as BOOLEAN) as PEPIsMaterial
  ,cast(r.ProductsApplicableRisk as BOOLEAN) as ProductsAndServicesIsMaterial
  ,cast(r.SectorApplicableRisk as BOOLEAN) as SectorIsMaterial
  ,cast(r.StructureApplicableRisk as BOOLEAN) as StructureIsMaterial
  ,cast(r.ThirdPartyApplicableRisk as BOOLEAN) as ThirdPartyIsMaterial
  ,cast(r.TransactionApplicableRisk as BOOLEAN) as TransactionIsMaterial
  ,cast(null as BOOLEAN) as OtherIsMaterial
from
  Legacy2_client c
    Inner join Legacy2_risk r
      on r.ClientId = c.ClientId
where
  c.IsClient = 'true'
  and c.ClientTypeId in (1, 2, 3)

In [0]:
# %sql
# CREATE OR REPLACE TEMPORARY VIEW NLSVF_Client AS
# select distinct
# concat('NLSVF_',c.CIFNUMBER) as LocalSystemIdentifier
# ,c.CIFNUMBER as ClientId
# ,c.CIFNUMBER as CaseId
# ,case when c.ENTITY = 'Individual' then concat('NP_NPPC_', c.CIFNUMBER) else concat('LEC_', c.CIFNUMBER) end as UniqueCaseId
# ,acc.userdef09 as GeographicalRiskLevel
# ,acc.userdef10 as EntityTypeRiskLevel
# ,acc.userdef11 as StructureRiskLevel
# ,acc.userdef12 as SectorRiskLevel
# ,acc.userdef13 as ProductAndServiceRiskLevel
# ,acc.userdef14 as PEPRiskLevel
# ,acc.userdef15 as TransactionRiskLevel
# ,acc.userdef16 as DistributionRiskLevel
# ,acc.userdef16 as ThirdPartyRiskLevel
# ,acc.userdef18 as AdverseInfoRiskLevel
# ,acc.userdef02 as CalculatedRiskLevel
# ,acc.userdef03 as RecalculatedRiskLevel
# from nls_dbo_cif c
# inner join nls_dbo_loanacct ac on c.CIFNO = ac.CIFNO
# inner join nls_dbo_loanacct_detail acc on ac.acctrefno = acc.acctrefno

MDM RANZ Risk Categories

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW RANZ_MaterialRisk AS
with QuestionLevelMateriality_material as 
(
select distinct
qa.InstanceId--,qa.SourceSystemReferenceId,qa.SourceSystemName,qa.MaterialQuestionId,qa.MaterialQuestionCode
,case when qa.MaterialQuestionCode like 'GEO%' then 'Geographical'
      when qa.MaterialQuestionCode like 'STR%' then 'Structure'
      when qa.MaterialQuestionCode like 'SEC%' then 'Sector'
      when qa.MaterialQuestionCode like 'TX%' then 'Transaction'
      when qa.MaterialQuestionCode like 'PS%' then 'Products and Services'
      when qa.MaterialQuestionCode like 'ADVR%' then 'Adverse Info'
      when qa.MaterialQuestionCode like 'ENT%' then 'Entity Type'
      when qa.MaterialQuestionCode like 'PEP%' then 'PEP'
      when qa.MaterialQuestionCode like '3RD%' then 'Third Party'
      when qa.MaterialQuestionCode like 'DIST%' then 'Distribution Channel'
      when qa.MaterialQuestionCode like 'OTH%' then 'Other'
  end as CategoryName
,qa.IsMaterial,qa.CaseStatusName, qa.SourceClient
from Party_RiskModelInstanceQuestionAnswers qa
where qa.SourceSystemName = 'RANZ - PegaCdd' and qa.ModelType = 'QuestionLevelMateriality'
)
,QuestionLevelMateriality AS
(
select distinct
InstanceId,SourceClient,
MAX(CASE WHEN CategoryName = 'Adverse Info' then IsMaterial else NULL END ) as AdverseInfoIsMaterial,
MAX(CASE WHEN CategoryName = 'Distribution Channel' then IsMaterial else NULL END ) as DistributionChannelIsMaterial,
MAX(CASE WHEN CategoryName = 'Entity Type' then IsMaterial else NULL END ) as EntityTypeIsMaterial,
MAX(CASE WHEN CategoryName = 'Geographical' then IsMaterial else NULL END ) as GeographicalIsMaterial,
MAX(CASE WHEN CategoryName = 'Other' then IsMaterial else NULL END ) as OtherIsMaterial,
MAX(CASE WHEN CategoryName = 'PEP' then IsMaterial else NULL END ) as PEPIsMaterial,
MAX(CASE WHEN CategoryName = 'Products and Services' then IsMaterial else NULL END ) as ProductsAndServicesIsMaterial,
MAX(CASE WHEN CategoryName = 'Sector' then IsMaterial else NULL END ) as SectorIsMaterial,
MAX(CASE WHEN CategoryName = 'Structure' then IsMaterial else NULL END ) as StructureIsMaterial,
MAX(CASE WHEN CategoryName = 'Third Party' then IsMaterial else NULL END ) as ThirdPartyIsMaterial,
MAX(CASE WHEN CategoryName = 'Transaction' then IsMaterial else NULL END ) as TransactionIsMaterial
FROM QuestionLevelMateriality_material
GROUP BY InstanceId,SourceClient
)
select * from QuestionLevelMateriality

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW RANZ_riskdetails AS
select distinct
CONCAT('RANZ_',p.PKEY_SRC_OBJECT) LocalSystemIdentifier,
p.PKEY_SRC_OBJECT as ClientID,
dd.CASE_ID as CaseID,
null as UniqueCaseId, 
 rc.GeographicalRiskLevel,
 rc.EntityTypeRiskLevel,
 rc.StructureRiskLevel,
 rc.SectorRiskLevel,
 rc.ProductAndServiceRiskLevel,
 rc.PEPRiskLevel,
 rc.TransactionRiskLevel,
 rc.DistributionRiskLevel,
 rc.ThirdPartyRiskLevel,
 rc.AdverseInfoRiskLevel,
  f.Description as CalculatedRiskLevel,
 g.Description as RecalculatedRiskLevel,
 rc.OtherRiskLevel,
 d.AdverseInfoIsMaterial,
 d.DistributionChannelIsMaterial,
 d.EntityTypeIsMaterial,
 d.GeographicalIsMaterial,
 d.PEPIsMaterial,
 d.ProductsAndServicesIsMaterial,
 d.SectorIsMaterial,
 d.StructureIsMaterial,
 d.ThirdPartyIsMaterial,
 d.TransactionIsMaterial,
 d.OtherIsMaterial
FROM c_b_party as p
JOIN c_b_contr_rol_party as crp
    ON crp.FK_PARTY_ID = p.ROWID_XREF
JOIN c_b_contract as c
    ON c.CONTR_ID = crp.FK_CONTR_ID
LEFT JOIN c_b_due_diligence as dd
    ON dd.FK_CONTR_ID = c.CONTR_ID
left join Party_RiskModelInstanceQuestionAnswers b
on dd.GRAM_INSTANCE_ID = b.InstanceId
 and b.SourceSystemName = 'RANZ - PegaCdd'
left join RiskModelCategories rc
on dd.GRAM_INSTANCE_ID = rc.InstanceId
left join RANZ_MaterialRisk d
on b.MaterialInstanceId = d.InstanceId
left join Party_RiskModelInstance e
on dd.GRAM_INSTANCE_ID = e.InstanceId
 and e.SourceSystemName = 'RANZ - PegaCdd'
 left join static_RiskLevel f
 on e.OverallCalculatedRiskLevelId = f.Id
  left join static_RiskLevel g
 on e.OverallRecalculatedRiskLevelId = g.Id

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Case_RiskCategory AS
select * FROM KN1_Risk
UNION
select * from gcob_CasesRisk
UNION
select * from Legacy2_riskdetails
UNION
select * from RANZ_riskdetails

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW Case_RiskCategory_Final AS
SELECT
    LocalSystemIdentifier,
    ClientId,
    CASE 
        WHEN CaseID RLIKE '^[0-9]+$' THEN CAST(CaseID AS INT)
        ELSE NULL
    END AS CaseId,
    CAST(CaseID AS STRING) AS CaseID_string,
    *
EXCEPT (LocalSystemIdentifier, ClientId, CaseId) 
FROM Case_RiskCategory;

In [0]:
df_party_CDDCases_RiskCategory=spark.table('Case_RiskCategory_Final')

In [0]:
df_final_party_CDDcase_RiskCategory=add_party_identifier(df_party_CDDCases_RiskCategory,df_Party_SystemIdentifier)

In [0]:
if RunType == "historical":
    save_to_saradar_storage_account(df_final_party_CDDcase_RiskCategory, party_CDDCases_RiskCategory_dataobject, radar_datamodel_version_number, environment,Load_Date)
else:
    save_to_saradar_storage_account(df_final_party_CDDcase_RiskCategory, party_CDDCases_RiskCategory_dataobject, radar_datamodel_version_number, environment)